# Question 10 - 45. Jump Game II

You are given a **0-indexed** array of integers `nums` of length `n`. You are initially positioned at `nums[0]`.

Each element `nums[i]` represents the maximum length of a forward jump from index `i`. In other words, if you are at `nums[i]`, you can jump to any `nums[i + j]` where:

- `0 <= j <= nums[i]` and
- `i + j < n`

Return the **minimum number of jumps** to reach `nums[n - 1]`. The test cases are generated such that you can reach `nums[n - 1]`.

**Example 1:**

    Input: nums = [2,3,1,1,4]
    Output: 2
    Explanation: The minimum number of jumps to reach the last index is 2. Jump 1 step from index 0 to 1, then 3 steps to the last index.

**Example 2:**

    Input: nums = [2,3,0,1,4]
    Output: 2

**Constraints:**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 1000`
- It's guaranteed that you can reach `nums[n - 1]`.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy / implicit BFS by levels (window of indices reachable in k jumps)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(2ⁿ) | O(n) |
| 1½ DP | O(n²) | O(n) |
| 2️⃣ Optimized | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"This is Jump Game I, but now the end is guaranteed reachable and I need the **fewest** jumps. 'Fewest steps' makes me think of shortest path, so BFS."*

- Reaching the end is guaranteed? → **Yes.** So no "impossible" case to handle.
- `n == 1`? → Already there → **0** jumps.
- Jump values are maximums (I can jump shorter)? → **Yes.** As in Jump Game I, reachable indices form contiguous blocks.
- Return just the count, not the path.

### Step 1 · Brute force

**Idea:** From index `i`, try every jump `1..nums[i]` and take `1 + min(jumps from landing spot)`.

🗣️ *"The brute force recursively tries every jump and takes the minimum. It's exponential because the same index is solved again from many paths."*

In [1]:
class SolutionBrute:
    def jump(self, nums: list[int]) -> int:
        last = len(nums) - 1

        def min_jumps(i: int) -> float:
            if i >= last:
                return 0
            best = float("inf")
            for step in range(1, nums[i] + 1):
                best = min(best, 1 + min_jumps(i + step))
            return best

        return min_jumps(0)

#### Step 1½ · DP: cache the answer per index

`dp[i]` = min jumps to reach index `i`. Push forward: from each `i`, every `j` in `i+1 … i+nums[i]` can be reached in `dp[i] + 1`.

🗣️ *"Caching gives O(n) states, but each pushes to up to `nums[i]` neighbours, so it's O(n·max(nums)), which is O(n²) in the worst case. Better, but I think the contiguity trick from Jump Game I gets this to linear."*

In [2]:
class SolutionDP:
    def jump(self, nums: list[int]) -> int:
        n = len(nums)
        dp = [float("inf")] * n
        dp[0] = 0
        for i in range(n):
            for j in range(i + 1, min(i + nums[i], n - 1) + 1):
                dp[j] = min(dp[j], dp[i] + 1)
        return dp[-1]

### Step 2 · Optimize

**Bottleneck:** The DP relaxes every edge `i → j`. There can be O(n²) edges.

**Key insight: think in BFS *levels*, not individual edges.**
- Level 0 = `{0}`.
- The indices reachable in **exactly ≤ k jumps** form a **contiguous window** `[start … end]`, since jumps are maximums.
- The next level's window ends at `farthest = max(i + nums[i])` over all `i` in the current window.

So I don't need a queue. A window is just two numbers. While scanning `i` left to right, I keep:
- `cur_end`: right edge of the window I can reach with `jumps` jumps.
- `farthest`: the farthest edge reachable with **one more** jump from anything seen so far.

When `i` hits `cur_end`, I've finished scanning this level. I *must* jump again: `jumps += 1`, `cur_end = farthest`.

🗣️ *"This is BFS on an implicit graph where each level is a contiguous range. I scan the current range, track how far the next range can reach, and when I hit the end of the current range I count a jump and move to the next range. Each index is touched once, so it's O(n) time and O(1) space."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Recursion over all jump choices | ❌ | Exponential. It re-solves the same suffix many times. |
| DP array `dp[i]` | ⚠️ | Correct and simple, but O(n²) edge relaxations and O(n) space. |
| Explicit BFS with `deque` + `visited` | ⚠️ | Right idea (fewest steps = BFS), but pushing every neighbour is still O(n²) edges, plus O(n) memory. |
| Dijkstra / heap | ❌ | Overkill. All edges have weight 1, so BFS is already optimal. A heap only adds a log factor. |
| **Greedy level-window (`cur_end`, `farthest`)** | ✅ | BFS where each level is compressed into a range. Each index is scanned once and there's no container. |

**Why greedy is correct:** every index in the current window costs the same number of jumps. So choosing *where* inside the window to jump from never matters. Only the farthest reach of the window does.

**Why loop to `n − 2`?** If `i == n − 1` happened to equal `cur_end`, we'd count an extra, useless jump *from* the last index.

In [3]:
class Solution:
    def jump(self, nums: list[int]) -> int:
        jumps = 0
        cur_end = 0      # right edge of indices reachable with `jumps` jumps
        farthest = 0     # right edge reachable with `jumps + 1` jumps
        for i in range(len(nums) - 1):   # never jump *from* the last index
            farthest = max(farthest, i + nums[i])
            if i == cur_end:             # finished this BFS level
                jumps += 1
                cur_end = farthest
                if cur_end >= len(nums) - 1:
                    break
        return jumps

### Step 3 · Toy example walkthrough

`nums = [2, 3, 1, 1, 4]`, last index = 4

| i | nums[i] | farthest = max(farthest, i+nums[i]) | i == cur_end? | jumps | cur_end |
|---|---|---|---|---|---|
| start | | 0 | | 0 | 0 |
| 0 | 2 | 2 | ✅ (0 == 0), level 0 done | **1** | 2 |
| 1 | 3 | **4** | ❌ | 1 | 2 |
| 2 | 1 | 4 | ✅ (2 == 2), level 1 done | **2** | 4 → ≥ last, break |

Answer **2**.

Seen as BFS levels:
- Level 0: `[0, 0]`
- Level 1 (1 jump): `[1, 2]`
- Level 2 (2 jumps): `[3, 4]` ← contains the last index

The window grows from `[0,0]` to `[1,2]` to `[3,4]`. Each window is just the old `cur_end + 1` up to `farthest`.

In [4]:
cases = [
    ([2, 3, 1, 1, 4], 2),
    ([2, 3, 0, 1, 4], 2),
    ([0], 0),                 # already at the end
    ([1, 2], 1),
    ([1, 1, 1, 1], 3),
    ([10, 1, 1, 1, 1], 1),
    ([1, 2, 1, 1, 1], 3),
]
for nums, expected in cases:
    assert SolutionBrute().jump(nums) == expected
    assert SolutionDP().jump(nums) == expected
    assert Solution().jump(nums) == expected
    print(nums, "->", expected)

# randomized cross-check (only on reachable inputs)
import random
for _ in range(500):
    nums = [random.randint(1, 3) for _ in range(random.randint(1, 10))]
    assert Solution().jump(nums) == SolutionBrute().jump(nums), nums
print("All tests passed ✅")

[2, 3, 1, 1, 4] -> 2
[2, 3, 0, 1, 4] -> 2
[0] -> 0
[1, 2] -> 1
[1, 1, 1, 1] -> 3
[10, 1, 1, 1, 1] -> 1
[1, 2, 1, 1, 1] -> 3
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Brute force | O(2ⁿ)-ish | O(n) stack | Re-solves suffixes |
| DP | O(n²) | O(n) | Relaxes every edge |
| **Greedy BFS levels** | **O(n)** | **O(1)** | Each level is a contiguous window, so two integers describe it |

**Complexity, explained:** the `for` loop visits each index once and does O(1) work per index. `cur_end` and `farthest` only ever move forward. We store three integers, so space is O(1).

**Edge cases:** `n == 1` → loop doesn't run → 0. Big first jump (`[10,…]`) → 1. All ones → `n − 1`.

**Follow-ups:**
- *What if the end might be unreachable?* → If `i > cur_end` at any point (or `farthest` doesn't grow past `i` when a level ends), return −1.
- *Return the actual path?* → Remember which `i` produced each level's `farthest`.
- *Connection:* Jump Game I asks "is the end inside the reachable set?". Jump Game II asks "how many levels until it is?".

🗣️ **30-second recap:** *"Fewest jumps is a shortest-path question, so BFS. Since each BFS level is a contiguous range of indices, I don't need a queue. I scan left to right keeping the current level's end and the farthest reach of the next level. Each time I hit the current end, I count one jump and move to the next level. O(n) time, O(1) space."*